In [ ]:
import pandas as pd
import glob
import os
import re

datasets = ['tax', 'hospital']

In [ ]:
# Read all mapping GT CSV files
csv_files = glob.glob('mapping_gt_results/llm_mapping_gt_*.csv')

# Load all dataframes
dfs = []
for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    dfs.append(df)

# Combine all dataframes
df_combined = pd.concat(dfs, ignore_index=True)

# Parse variant_name to extract model and prompt_key
# Format: "Model DIRECT (prompt_key)"
def parse_variant_name(variant_name):
    if pd.isna(variant_name) or variant_name == 'N/A':
        return None, None
    
    # Match pattern: "Model DIRECT (prompt_key)"
    match = re.match(r'^(.+?)\s+DIRECT\s+\((.+?)\)$', variant_name)
    if match:
        model = match.group(1).strip()
        prompt_key = match.group(2).strip()
        return model, prompt_key
    return None, None

# Parse and add columns
parsed = df_combined['variant_name'].apply(parse_variant_name)
df_combined['model'] = [p[0] for p in parsed]
df_combined['prompt_key'] = [p[1] for p in parsed]

# Filter out rows where parsing failed
df_combined = df_combined[df_combined['model'].notna()].copy()

print(f"Total rows: {len(df_combined)}")
print(f"Datasets: {sorted(df_combined['dataset'].unique())}")
print(f"Models: {sorted(df_combined['model'].unique())}")
print(f"Prompt keys: {sorted(df_combined['prompt_key'].unique())}")
print(f"Modes: {sorted(df_combined['mode'].unique())}")
print()

# Simplify model names for display
model_display_names = {
    'GPT-5-mini': 'GPT-5-mini',
    'GPT-5': 'GPT-5',
    'Mistral-small': 'Mistral-Small',
    'Mistral-large': 'Mistral-Large',
    'Llama-3.3-70b': 'Llama-3.3-70b'
}

# Prompt key display names
prompt_display_names = {
    'zero_shot': 'Zero-shot',
    'zero_shot_with_table': 'Zero-shot+Table',
    'with_examples': 'Few-shot',
    'with_examples_and_table': 'Few-shot+Table'
}

# Add readable columns
df_combined['model_display'] = df_combined['model'].map(model_display_names).fillna(df_combined['model'])
df_combined['prompt_display'] = df_combined['prompt_key'].map(prompt_display_names).fillna(df_combined['prompt_key'])

print("\n" + "="*120)
print("MAPPING RESULTS ANALYSIS - RANKING BY CONSTRAINT-LEVEL F1")
print("="*120)
print("\n📊 Available Metrics:")
print("  • Constraint Level: exact match precision/recall/F1")
print("  • Pairwise Syntactic: exact match & Jaccard similarity")
print("  • Pairwise Violation: exact match & Jaccard similarity")
print("  • Aggregate Violation: precision/recall/F1")
print("="*120)

Total rows: 108
Datasets: <ArrowStringArray>
['sisu', 'hospital', 'restaurant', 'tax', 'person']
Length: 5, dtype: str
Models: <ArrowStringArray>
[     'mistral-small-2506',              'gpt-5-mini',
                   'gpt-5',      'mistral-large-2512',
 'llama-3.3-70b-versatile',            'sabiazinho-4',
                 'sabia-4']
Length: 7, dtype: str
Prompt keys: <ArrowStringArray>
[                 'zero_shot',       'zero_shot_with_table',
              'with_examples',    'with_examples_and_table',
               'zero_shot_pt',    'zero_shot_with_table_pt',
           'with_examples_pt', 'with_examples_and_table_pt']
Length: 8, dtype: str

Generating tables for each dataset...

HOSPITAL DATASET:
\begin{table}[htbp]
\centering
\small
\caption{Sentence-level rule classification on the Hospital dataset. All variants include a data quality constraint definition; ``Examples'' denotes whether generic data-quality examples were added, and ``Context'' denotes whether table schema a

In [ ]:
# PER-DATASET ANALYSIS: Best configurations per dataset
print("\n" + "="*120)
print("RANKING REPORT: BEST CONFIGURATIONS PER DATASET")
print("Showing top configurations ranked by Constraint-Level F1")
print("="*120)

for dataset in sorted(df_combined['dataset'].unique()):
    df_dataset = df_combined[df_combined['dataset'] == dataset].copy()
    
    # Sort by constraint F1 (descending)
    df_dataset = df_dataset.sort_values('constraint_exact_f1', ascending=False)
    
    print(f"\n📊 {dataset.upper()} DATASET")
    print("-" * 120)
    
    # Create detailed display dataframe with all metric levels
    display_df = pd.DataFrame({
        'Model': df_dataset['model_display'],
        'Prompt': df_dataset['prompt_display'],
        # Constraint level
        'Const F1': df_dataset['constraint_exact_f1'].round(4),
        'Const Prec': df_dataset['constraint_exact_precision'].round(4),
        'Const Rec': df_dataset['constraint_exact_recall'].round(4),
        # Pairwise syntactic
        'PW Synt Match': df_dataset['pairwise_syntactic_exact_match'].round(4),
        'PW Synt Jacc': df_dataset['pairwise_syntactic_predicate_jaccard'].round(4),
        # Pairwise violation
        'PW Viol Match': df_dataset['pairwise_violation_exact_match'].round(4),
        'PW Viol Jacc': df_dataset['pairwise_violation_jaccard'].round(4),
        # Aggregate violation
        'Agg Viol F1': df_dataset['aggregate_violation_f1'].round(4),
        'Agg Viol Prec': df_dataset['aggregate_violation_precision'].round(4),
        'Agg Viol Rec': df_dataset['aggregate_violation_recall'].round(4),
    })
    
    display_df = display_df.reset_index(drop=True)
    display_df.index = display_df.index + 1  # Start ranking from 1
    display_df.index.name = 'Rank'
    
    # Show top 10
    print(display_df.head(10).to_string())
    print()
    
    # Show top 3 summary
    print(f"🏆 Top 3 for {dataset} (by Constraint F1):")
    for i, (idx, row) in enumerate(display_df.head(3).iterrows(), 1):
        print(f"  {i}. {row['Model']:15s} | {row['Prompt']:15s} | Const F1: {row['Const F1']:.4f} | Agg Viol F1: {row['Agg Viol F1']:.4f}")
    print()

print("="*120)

In [ ]:
# CROSS-DATASET ANALYSIS: Best configuration across all datasets
print("\n" + "="*120)
print("🌟 CROSS-DATASET RANKING: BEST OVERALL CONFIGURATIONS")
print("Showing configurations with HIGHEST AVERAGE CONSTRAINT-LEVEL RECALL, ranked by Avg Constraint F1")
print("="*120)

# Group by configuration (model, prompt_key)
grouped = df_combined.groupby(['model', 'prompt_key']).agg({
    'constraint_exact_precision': 'mean',
    'constraint_exact_recall': 'mean',
    'constraint_exact_f1': 'mean',
    'pairwise_syntactic_exact_match': 'mean',
    'pairwise_syntactic_predicate_jaccard': 'mean',
    'pairwise_violation_exact_match': 'mean',
    'pairwise_violation_jaccard': 'mean',
    'aggregate_violation_precision': 'mean',
    'aggregate_violation_recall': 'mean',
    'aggregate_violation_f1': 'mean',
}).reset_index()

# Rename columns to be clear they are averages
grouped.columns = [
    'model', 'prompt_key', 
    'avg_const_prec', 'avg_const_rec', 'avg_const_f1',
    'avg_pw_synt_match', 'avg_pw_synt_jacc',
    'avg_pw_viol_match', 'avg_pw_viol_jacc',
    'avg_agg_viol_prec', 'avg_agg_viol_rec', 'avg_agg_viol_f1'
]

# Add display names
grouped['model_display'] = grouped['model'].map(model_display_names).fillna(grouped['model'])
grouped['prompt_display'] = grouped['prompt_key'].map(prompt_display_names).fillna(grouped['prompt_key'])

# Find maximum average constraint recall
max_avg_recall = grouped['avg_const_rec'].max()

print(f"\n📈 Maximum Average Constraint Recall: {max_avg_recall:.4f}")

# Filter to configurations with maximum average recall
best_configs = grouped[grouped['avg_const_rec'] == max_avg_recall].copy()

# Sort by average constraint F1 score (descending)
best_configs = best_configs.sort_values('avg_const_f1', ascending=False).reset_index(drop=True)

print(f"Found {len(best_configs)} configuration(s) achieving maximum average constraint recall\n")

# Create clean display dataframe
display_cross = pd.DataFrame({
    'Model': best_configs['model_display'],
    'Prompt': best_configs['prompt_display'],
    'Avg Const F1': best_configs['avg_const_f1'].round(4),
    'Avg Const Prec': best_configs['avg_const_prec'].round(4),
    'Avg Const Rec': best_configs['avg_const_rec'].round(4),
    'Avg PW Synt': best_configs['avg_pw_synt_match'].round(4),
    'Avg PW Viol': best_configs['avg_pw_viol_match'].round(4),
    'Avg Agg Viol F1': best_configs['avg_agg_viol_f1'].round(4),
})

display_cross.index = display_cross.index + 1  # Start ranking from 1
display_cross.index.name = 'Rank'

print(display_cross.to_string())
print()

# Show the winner
if len(best_configs) > 0:
    winner = best_configs.iloc[0]
    print("="*120)
    print("🏆 BEST OVERALL CONFIGURATION:")
    print("="*120)
    print(f"  Model:                  {winner['model_display']}")
    print(f"  Prompt:                 {winner['prompt_display']}")
    print(f"  Avg Constraint Recall:  {winner['avg_const_rec']:.4f}")
    print(f"  Avg Constraint Prec:    {winner['avg_const_prec']:.4f}")
    print(f"  Avg Constraint F1:      {winner['avg_const_f1']:.4f}")
    print(f"  Avg PW Synt Match:      {winner['avg_pw_synt_match']:.4f}")
    print(f"  Avg PW Viol Match:      {winner['avg_pw_viol_match']:.4f}")
    print(f"  Avg Agg Viol F1:        {winner['avg_agg_viol_f1']:.4f}")
    print("="*120)

    # Show detailed breakdown per dataset for the winner
    print(f"\n📋 Detailed Performance of Best Configuration on Each Dataset:")
    print("-" * 120)
    
    winner_details = df_combined[
        (df_combined['model'] == winner['model']) &
        (df_combined['prompt_key'] == winner['prompt_key'])
    ].copy()
    
    winner_details_display = pd.DataFrame({
        'Dataset': winner_details['dataset'],
        'Const F1': winner_details['constraint_exact_f1'].round(4),
        'Const Prec': winner_details['constraint_exact_precision'].round(4),
        'Const Rec': winner_details['constraint_exact_recall'].round(4),
        'PW Synt': winner_details['pairwise_syntactic_exact_match'].round(4),
        'PW Viol': winner_details['pairwise_violation_exact_match'].round(4),
        'Agg Viol F1': winner_details['aggregate_violation_f1'].round(4),
    })
    
    winner_details_display = winner_details_display.sort_values('Dataset').reset_index(drop=True)
    winner_details_display.index = winner_details_display.index + 1
    print(winner_details_display.to_string())
    print()
    
    # Summary statistics
    print("📊 Summary Statistics for Winner:")
    print(f"  Best Dataset (Const F1):  {winner_details_display.loc[winner_details_display['Const F1'].idxmax(), 'Dataset']} (F1: {winner_details_display['Const F1'].max():.4f})")
    print(f"  Worst Dataset (Const F1): {winner_details_display.loc[winner_details_display['Const F1'].idxmin(), 'Dataset']} (F1: {winner_details_display['Const F1'].min():.4f})")
    print(f"  Const F1 Range: {winner_details_display['Const F1'].min():.4f} - {winner_details_display['Const F1'].max():.4f}")
    print(f"  Const F1 Std Dev: {winner_details_display['Const F1'].std():.4f}")
    print("\n" + "="*120)

In [ ]:
# COMPARISON: Top 5 Configurations Across Datasets
print("\n" + "="*120)
print("📊 COMPARISON: Top 5 Configurations Across Datasets")
print("="*120)

# Get top 5 by average constraint F1 among those with max recall
top_5 = best_configs.head(5)

for idx, config in top_5.iterrows():
    config_details = df_combined[
        (df_combined['model'] == config['model']) &
        (df_combined['prompt_key'] == config['prompt_key'])
    ].copy()
    
    print(f"\n#{idx}: {config['model_display']} | {config['prompt_display']}")
    print("-" * 120)
    
    details_display = pd.DataFrame({
        'Dataset': config_details['dataset'],
        'Const F1': config_details['constraint_exact_f1'].round(4),
        'Const Prec': config_details['constraint_exact_precision'].round(4),
        'Const Rec': config_details['constraint_exact_recall'].round(4),
        'PW Synt Match': config_details['pairwise_syntactic_exact_match'].round(4),
        'PW Viol Match': config_details['pairwise_violation_exact_match'].round(4),
        'Agg Viol F1': config_details['aggregate_violation_f1'].round(4),
    })
    
    details_display = details_display.sort_values('Dataset')
    print(details_display.to_string(index=False))
    
    avg_const_f1 = config_details['constraint_exact_f1'].mean()
    print(f"  → Average Constraint F1: {avg_const_f1:.4f}")

print("\n" + "="*120)